# Deep Learning Multi-Seed Replication & Light Hyperparameter Tuning (RSIC v6)
**Pre-Experimental Protocol & Replication Notebook**

## Aturan Keras & Metodologi Pre-Eksperimen:
1. **Pre-Experimental Design & Pemilihan Loss**:
   - Seluruh desain, grid hyperparameter (termasuk dimensi loss MAE vs MSE), model, horizon, dan keluarga pengujian hipotesis (Bonferroni) ditetapkan **SEBELUM** melihat hasil empiris run v6.
   - Pemilihan hyperparameter dan loss murni ditetapkan dari validasi (167 observasi), tanpa melihat test set (335 cutoff out-of-sample).
   - Loss function yang dipantau pada early stopping (`ptl/val_loss`) menggunakan satuan fungsi loss pelatihan masing-masing (MAE atau MSE).
2. **Tanpa Kebocoran Data (Strict Anti-Leakage Guard & Koreksi Batas Fit/Val/Test)**:
   - Evaluasi out-of-sample pada test set memiliki 335 cutoff rolling dengan horizon $h=10$, yang mencakup $335 + 10 - 1 = 344$ baris target unik (bukan 335).
   - Data pelatihan (`fit`) pada Stage C ditegaskan secara ketat hanya menggunakan baris 1 s.d. 1332 (1676 - 344 = 1332 baris, terdiri dari 1165 Train + 167 Val).
   - Split sebenarnya adalah **1165 Train / 167 Val / 344 Baris Target (335 cutoff rolling)**.
   - Tiga assert anti-leakage wajib:
     (a) `max real_date data fit == tanggal cutoff pertama` (2025-04-23).
     (b) `tanggal target pertama window pertama > max real_date data fit` (2025-04-24 > 2025-04-23).
     (c) Pada hasil cross_validation smoke test, `min ds target > max ds data fit`.
3. **Pemberantasan Pelatihan Ganda & Verifikasi Bobot**:
   - Model dilatih tepat SATU kali per (konfigurasi, seed) menggunakan `NeuralForecast.fit()`.
   - Evaluasi rolling out-of-sample dijalankan secara murni inferensial melalui `NeuralForecast.cross_validation(..., use_fitted=True)`.
   - Integritas bobot dibuktikan via perbandingan checksum MD5 bobot sebelum dan sesudah cross-validation (`chk_before == chk_after`).
4. **Restore Best Weights & Early Stopping Aktual**:
   - Callback bawaan NeuralForecast (`EarlyStopping`) tidak boleh ditimpa; callback tambahan selalu di-append/extend.
   - Early stopping memonitor `ptl/val_loss` dengan `val_check_steps=25` dan `early_stop_patience_steps=5`.
   - Didukung oleh callback in-memory `RestoreBestWeightsCallback` yang mengabaikan evaluasi validasi sanity-check / step 0 (agar bobot acak tidak pernah menjadi bobot terbaik).
   - Checksum bobot akhir model terbukti matematis identik dengan checksum pada `best_step` (dengan `best_step > 0`).
   - Titik henti `actual_final_step`, `best_step`, `best_val_loss`, dan status `es_triggered` dicatat ke `v6_es_log.csv`.
5. **Multi-Seed Replication**: Dievaluasi pada 5 seed deterministik: `[42, 7, 13, 21, 99]`.
6. **Keluarga Bonferroni A Priori & Transparansi Dua Seri Hist_Exp**:
   - Komparasi DL Ensemble vs Baselines (`Hist_Exp`, `Roll_20`, `HAR`): $m = 12$ per horizon ($4 \text{ konfigurasi} \times 3 \text{ baseline}$).
   - Uji Ablasi Ensemble (`SPILL` vs `NOSPILL`): $m = 4$ ($2 \text{ model} \times 2 \text{ horizon}$).
   - Seri `Hist_Exp` didefinisikan secara transparan:
     - Seri DL-base (MSE $h=1$: $2.689570 \times 10^{-6}$, $h=10$: $2.686277 \times 10^{-6}$) dipakai untuk evaluasi model DL.
     - Seri HAR-file (MSE $h=1$: $2.689342 \times 10^{-6}$, $h=10$: $2.686052 \times 10^{-6}$) diverifikasi pada reproduksi benchmark HAR v2.
     - Penyebab selisih: baris 2020-01-03 terbuang di df_har oleh `dropna()` setelah left-merge dengan file TVP.


# CELL 1: SETUP LINGKUNGAN, PINNING NEURALFORECAST 3.3.0, GLOBAL SEED & AUDIT GPU


In [1]:
import os
import sys
import subprocess
import random
import hashlib
import json
import time
import glob
import zipfile
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

# 1. Pengecekan Lingkungan & Pinning neuralforecast==3.3.0
print("=== CELL 1: SETUP LINGKUNGAN, PINNING NF 3.3.0, GLOBAL SEED & AUDIT GPU ===")
print(f"Python Version   : {sys.version.split()[0]}")

# Deteksi koneksi internet
has_net = False
try:
    import socket
    socket.setdefaulttimeout(3)
    socket.socket(socket.AF_INET, socket.SOCK_STREAM).connect(('8.8.8.8', 53))
    has_net = True
except Exception:
    has_net = False
print(f"Status Internet  : {'ONLINE' if has_net else 'OFFLINE'}")

# Pinning neuralforecast==3.3.0
try:
    import neuralforecast
    nf_ver = neuralforecast.__version__
except ImportError:
    nf_ver = None

if has_net and (os.path.exists('/kaggle') or (nf_ver != '3.3.0') or (nf_ver is None)):
    print("Menginstal/menyelaraskan paket neuralforecast==3.3.0...")
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'neuralforecast==3.3.0', 'arch', 'statsmodels'], check=True)
    import neuralforecast
    nf_ver = neuralforecast.__version__

import torch
import pytorch_lightning as pl
from neuralforecast import NeuralForecast
from neuralforecast.models import TFT, NBEATSx
from neuralforecast.losses.pytorch import MAE, MSE

print(f"PyTorch Version  : {torch.__version__}")
print(f"Lightning Version: {pl.__version__}")
print(f"NeuralForecast   : {nf_ver}")

# Assert versi wajib neuralforecast 3.3.0
assert nf_ver == '3.3.0', f"ASSERT STOP: neuralforecast harus versi 3.3.0, ditemukan {nf_ver}!"

# Deteksi GPU & Informasi Perangkat
gpu_available = torch.cuda.is_available()
gpu_name = torch.cuda.get_device_name(0) if gpu_available else "Tidak terdeteksi (CPU)"
print(f"GPU Terdeteksi   : {gpu_available} ({gpu_name})")

# 2. Fungsi Global Seed Deterministik
def set_all_seeds(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    pl.seed_everything(seed, workers=True)
    os.environ['PYTHONHASHSEED'] = str(seed)

set_all_seeds(42)
print("✅ Lingkungan tervalidasi dan seed awal 42 aktif.")


=== CELL 1: SETUP LINGKUNGAN, PINNING NF 3.3.0, GLOBAL SEED & AUDIT GPU ===
Python Version   : 3.13.15
Status Internet  : ONLINE
Menginstal/menyelaraskan paket neuralforecast==3.3.0...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.3/331.3 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 982.9/982.9 kB 23.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 348.7/348.7 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.4/50.4 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 3.7 MB/s eta 0:00:00
PyTorch Version  : 2.11.0+cu128
Lightning Version: 2.6.6
NeuralForecast   : 3.3.0


Seed set to 42


GPU Terdeteksi   : True (Tesla T4)
✅ Lingkungan tervalidasi dan seed awal 42 aktif.


# CELL 2: KONFIGURASI GLOBAL, SEEDS, MODELS, FEATURE SETS, & KELUARGA BONFERRONI


In [2]:
# ======================================================================================
# CELL 2: KONFIGURASI GLOBAL, SEEDS, MODELS, FEATURE SETS, & KELUARGA BONFERRONI
# ======================================================================================

STAGE = "ALL"  # <--- GANTI DI SINI: "A", "B", "C", "D", atau "ALL"

# 2. Daftar Seed Multi-Seed
SEEDS = [42, 7, 13, 21, 99]

# 3. Model & Feature Sets
MODELS = ['TFT', 'NBEATSx']

# 1. NOSPILL (11 fitur): 8 return/weekend + 3 kalender
NOSPILL_COLS = [
    'ret_saham_nvda', 'ret_saham_msft', 'ret_saham_tsla', 'ret_saham_googl', 'ret_saham_meta',
    'ret_crypto_btc_usd', 'ret_crypto_eth_usd', 'crypto_btc_usd_weekend_return',
    'gap_days', 'is_monday', 'is_ihsg_holiday'
]

# 2. SPILL (20 fitur): 11 NOSPILL + 9 spillover kausal murni dari TVP-VAR
SPILL_COLS = NOSPILL_COLS + [
    'spillover_ai_to_btcidr', 'tvp_tsi',
    'net_spill_saham_nvda', 'net_spill_saham_msft', 'net_spill_saham_tsla', 'net_spill_saham_googl', 'net_spill_saham_meta',
    'net_spill_crypto_btc_usd', 'net_spill_crypto_eth_usd'
]

FEATURE_SETS = {
    'NOSPILL': NOSPILL_COLS,
    'SPILL': SPILL_COLS
}

CONFIGS = [
    ('TFT', 'SPILL', 'T0'),
    ('TFT', 'NOSPILL', 'T0'),
    ('NBEATSx', 'SPILL', 'T0'),
    ('NBEATSx', 'NOSPILL', 'T0')
]

# 4. Parameter Latihan Standar
H = 10                     # Horizon peramalan (h=1 s.d. 10)
MAX_STEPS = 1000           # Budget iterasi maksimum
VAL_CHECK_STEPS = 25       # Frekuensi evaluasi validasi (setiap 25 steps)
CHOSEN_PATIENCE = 5        # 5 validation checks x 25 steps = 125 steps patience window

# 5. Keluarga Pengujian Hipotesis Bonferroni A Priori
M_FAMILY_BASELINES = len(CONFIGS) * 3  # 12
M_FAMILY_ABLATION = len(MODELS) * 2    # 4

print("=== KONFIGURASI EKSPERIMEN V6 ===")
print(f"Stage Aktif         : {STAGE}")
print(f"Seeds ({len(SEEDS)})        : {SEEDS}")
print(f"Configs ({len(CONFIGS)})      : {CONFIGS}")
print(f"Early Stopping      : patience={CHOSEN_PATIENCE} checks x val_check={VAL_CHECK_STEPS} steps ({CHOSEN_PATIENCE*VAL_CHECK_STEPS} steps)")
print(f"Bonferroni Families : Baselines m={M_FAMILY_BASELINES} per horizon | Ablation m={M_FAMILY_ABLATION}")


=== KONFIGURASI EKSPERIMEN V6 ===
Stage Aktif         : ALL
Seeds (5)        : [42, 7, 13, 21, 99]
Configs (4)      : [('TFT', 'SPILL', 'T0'), ('TFT', 'NOSPILL', 'T0'), ('NBEATSx', 'SPILL', 'T0'), ('NBEATSx', 'NOSPILL', 'T0')]
Early Stopping      : patience=5 checks x val_check=25 steps (125 steps)
Bonferroni Families : Baselines m=12 per horizon | Ablation m=4


# CELL 3: PEMUATAN DATA, VALIDASI MD5 HASH, MERGE TVP, SCHEMA, & KOREKSI SPLIT 1165/167/344


In [3]:
# ======================================================================================
# CELL 3: PEMUATAN DATA, VALIDASI MD5 HASH, MERGE TVP, SCHEMA, & KOREKSI SPLIT 1165/167/344
# ======================================================================================

# 1. Lokasi Direktori Dataset Input & Output
BASE_DIR = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.path.abspath(".")
SRC_DIR = os.path.join(BASE_DIR, "src")

CANDIDATE_DATASET_DIRS = [
    '/kaggle/input/datasets/ferastoro/spillover-v5-inputs/rsic-v5-inputs',
    os.path.join(BASE_DIR, "inputs", "v5"),
    "inputs/v5",
    "../inputs/v5",
    "rsic-v5-inputs",
    "../rsic-v5-inputs"
]
DATASET_DIR = None
for d in CANDIDATE_DATASET_DIRS:
    if os.path.exists(d) and os.path.exists(os.path.join(d, 'dataset_deep_learning.csv')):
        DATASET_DIR = d
        break

if DATASET_DIR is None:
    raise FileNotFoundError(f"Direktori dataset input tidak ditemukan di kandidat: {CANDIDATE_DATASET_DIRS}")

for p in [SRC_DIR, DATASET_DIR]:
    if os.path.exists(p) and p not in sys.path:
        sys.path.append(p)
import v5_helpers

OUTPUT_DIR = "/kaggle/working" if os.path.exists("/kaggle") else os.path.join(BASE_DIR, "results", "v6")
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Direktori Input : {DATASET_DIR}")
print(f"Direktori Output: {OUTPUT_DIR}")

# 2. Audit MD5 Hash File Input
def get_file_md5(p):
    if not os.path.exists(p):
        return "NOT_FOUND"
    h = hashlib.md5()
    with open(p, 'rb') as fp:
        while chunk := fp.read(8192):
            h.update(chunk)
    return h.hexdigest()

expected_hashes = {
    'dataset_deep_learning.csv': 'ba64297407f9d8221488b4815a12ce8c',
    'tvp_var_dynamic_features_causal.csv': '066ad3cdf980cd187a79c25dc210f1b2',
    'v5_helpers.py': 'bdd7bf54c3cca56f30a7a1cd779f65b1',
    'har_preds_T0_h1_v2.csv': 'e340a7ba8effd60c8c6d7b85b073aded',
    'har_preds_T0_h10_v2.csv': '43baf149f5f60be52a80b962ab93e481',
    'CV_Predictions_V4.csv': '6251bfb6ce543ffcd746e5fab98c3a63',
    'GARCH_Predictions_V4.csv': '5e84fa827fd5cfd13bea445230daf2ee'
}

print("\n=== AUDIT MD5 INPUT DATASET ===")
for fname, exp_h in expected_hashes.items():
    fpath = os.path.join(SRC_DIR, fname) if (fname == "v5_helpers.py" and not os.path.exists(os.path.join(DATASET_DIR, fname)) and os.path.exists(os.path.join(SRC_DIR, fname))) else os.path.join(DATASET_DIR, fname)
    act_h = get_file_md5(fpath)
    status = "PASS" if act_h == exp_h else f"FAIL (Found: {act_h})"
    print(f"File {fname:<36} | MD5: {act_h} | Status: {status}")
    assert act_h == exp_h, f"ASSERT FAILED: Hash {fname} tidak cocok!"

# 3. Pemuatan & Merge Causal TVP Features
df_dl = pd.read_csv(os.path.join(DATASET_DIR, 'dataset_deep_learning.csv'))
df_tvp = pd.read_csv(os.path.join(DATASET_DIR, 'tvp_var_dynamic_features_causal.csv'))

df_dl['Date'] = pd.to_datetime(df_dl['Date'])
df_tvp['Date'] = pd.to_datetime(df_tvp['Date'])

# hist_series historis 1677 baris dari df_dl (ret**2, index Date)
hist_series = pd.Series(df_dl['ret_crypto_btc_idr_indodax'].values**2, index=df_dl['Date'])

# Merge TVP Features
df_merged = pd.merge(df_dl, df_tvp, on='Date', how='left').sort_values('Date')
tvp_cols = [c for c in df_tvp.columns if c != 'Date']
df_merged[tvp_cols] = df_merged[tvp_cols].ffill()
df = df_merged.dropna().reset_index(drop=True)

df['volatility_btc_idr'] = df['ret_crypto_btc_idr_indodax'] ** 2

assert len(df) == 1676, f"ASSERT STOP: Total baris df harus 1676, ditemukan {len(df)}!"
assert len(SPILL_COLS) == 20, f"ASSERT STOP: Fitur SPILL harus 20, ditemukan {len(SPILL_COLS)}!"
assert len(NOSPILL_COLS) == 11, f"ASSERT STOP: Fitur NOSPILL harus 11, ditemukan {len(NOSPILL_COLS)}!"
assert 'net_spill_crypto_btc_idr_indodax' not in SPILL_COLS, "LEAKAGE: Target spillover dilarang ada di fitur!"
assert all(c in df.columns for c in SPILL_COLS), "Fitur SPILL tidak lengkap di DataFrame!"

# 4. Pembentukan Format Long NeuralForecast (Nixtla)
nf_df = pd.DataFrame()
nf_df['ds'] = pd.date_range(start='2020-01-01', periods=len(df), freq='D')
nf_df['unique_id'] = 'BTC_IDR'
nf_df['y'] = df['volatility_btc_idr'].values.astype('float32')
for col in sorted(SPILL_COLS):
    nf_df[col] = df[col].values.astype('float32')
nf_df['real_date'] = df['Date'].values

# 5. Split Kausal Presisi: 1165 Train / 167 Val / 344 Baris Target (335 cutoff test)
# 335 cutoffs rolling dengan h=10 memerlukan 335 + 10 - 1 = 344 baris target (baris 1332 s.d. 1675).
# Seluruh proses fit (Train + Val) harus berakhir tepat di baris 1676 - 344 = 1332 (baris 0 s.d. 1331).
N = len(nf_df)
n_test_cutoffs = 335
n_test_target_rows = n_test_cutoffs + H - 1  # 335 + 10 - 1 = 344
n_val = 167
n_train_val = N - n_test_target_rows          # 1676 - 344 = 1332
n_train = n_train_val - n_val                 # 1332 - 167 = 1165

assert n_train == 1165, f"ASSERT STOP: Ukuran train {n_train} != 1165!"
assert n_train_val == 1332, f"ASSERT STOP: Ukuran train_val {n_train_val} != 1332!"
assert n_test_target_rows == 344, f"ASSERT STOP: Ukuran target test {n_test_target_rows} != 344!"

df_train = nf_df.iloc[:n_train].copy()
df_val = nf_df.iloc[n_train:n_train_val].copy()
df_test = nf_df.iloc[n_train_val:].copy()
df_train_val = nf_df.iloc[:n_train_val].copy()

print(f"✅ Data siap: Total={N} baris | Train={len(df_train)} (1165) | Val={len(df_val)} (167) | Test Target={len(df_test)} (344 baris, {n_test_cutoffs} cutoffs)")


Direktori Input : /kaggle/input/datasets/ferastoro/spillover-v5-inputs/rsic-v5-inputs
Direktori Output: ./v6_outputs

=== AUDIT MD5 INPUT DATASET ===
File dataset_deep_learning.csv            | MD5: ba64297407f9d8221488b4815a12ce8c | Status: PASS
File tvp_var_dynamic_features_causal.csv  | MD5: 066ad3cdf980cd187a79c25dc210f1b2 | Status: PASS
File v5_helpers.py                        | MD5: bdd7bf54c3cca56f30a7a1cd779f65b1 | Status: PASS
File har_preds_T0_h1_v2.csv               | MD5: e340a7ba8effd60c8c6d7b85b073aded | Status: PASS
File har_preds_T0_h10_v2.csv              | MD5: 43baf149f5f60be52a80b962ab93e481 | Status: PASS
File CV_Predictions_V4.csv                | MD5: 6251bfb6ce543ffcd746e5fab98c3a63 | Status: PASS
File GARCH_Predictions_V4.csv             | MD5: 5e84fa827fd5cfd13bea445230daf2ee | Status: PASS
✅ Data siap: Total=1676 baris | Train=1165 (1165) | Val=167 (167) | Test Target=344 (344 baris, 335 cutoffs)


# CELL 4: STAGE A - AUDIT CV4 DATES, BENCHMARK HAR v2, STRICT DM ASSERT & HIST_EXP TRANSPARENCY


In [4]:
# ======================================================================================
# CELL 4: STAGE A - AUDIT CV4 DATES, BENCHMARK HAR v2, STRICT DM ASSERT & HIST_EXP TRANSPARENCY
# ======================================================================================
from sklearn.metrics import mean_squared_error

print("=== STAGE A: AUDIT PROVENANCE & CROSS-CHECK BENCHMARK (CPU) ===")

# 1. Audit Tanggal dan Dimensi pada CV_Predictions_V4.csv (Poin 1)
cv_v4_file = os.path.join(DATASET_DIR, 'CV_Predictions_V4.csv')
assert os.path.exists(cv_v4_file), f"File {cv_v4_file} tidak ditemukan!"
df_cv4 = pd.read_csv(cv_v4_file)
df_cv4['cutoff'] = pd.to_datetime(df_cv4['cutoff'])
df_cv4['real_date'] = pd.to_datetime(df_cv4['real_date'])
dup_count = df_cv4.duplicated(subset=['cutoff', 'real_date']).sum()
print(f"Duplikat [cutoff, real_date] di CV_Predictions_V4: {dup_count} (PASS: harus 0)")
assert dup_count == 0, "Ditemukan duplikasi baris pada cutoff dan real_date!"

# Audit Cutoffs & Target Dates dari CV_Predictions_V4.csv
unique_cutoffs_cv4 = df_cv4['cutoff'].drop_duplicates().sort_values()
unique_targets_cv4 = df_cv4['real_date'].drop_duplicates().sort_values()
cutoff_origins = v5_helpers.get_cutoff_origins(df_cv4, hist_series)

first_cutoff_syn = unique_cutoffs_cv4.iloc[0].strftime('%Y-%m-%d')
last_cutoff_syn = unique_cutoffs_cv4.iloc[-1].strftime('%Y-%m-%d')
first_cutoff_origin = list(cutoff_origins.values())[0].strftime('%Y-%m-%d')
last_cutoff_origin = list(cutoff_origins.values())[-1].strftime('%Y-%m-%d')

first_target_real = unique_targets_cv4.iloc[0].strftime('%Y-%m-%d')
last_target_real = unique_targets_cv4.iloc[-1].strftime('%Y-%m-%d')

n_cutoffs_act = len(unique_cutoffs_cv4)
n_targets_act = len(unique_targets_cv4)

print("\n--- AUDIT CV_PREDICTIONS_V4.CSV (Poin 1) ---")
print(f"Jumlah Cutoff Rolling                : {n_cutoffs_act} (Harapan: 335)")
print(f"Jumlah Tanggal Target Unik           : {n_targets_act} (Harapan: 344)")
print(f"Tanggal Cutoff Pertama (ds sintetis) : {first_cutoff_syn}")
print(f"Tanggal Cutoff Terakhir (ds sintetis): {last_cutoff_syn}")
print(f"Tanggal Cutoff Pertama (real_date)   : {first_cutoff_origin}")
print(f"Tanggal Cutoff Terakhir (real_date)  : {last_cutoff_origin}")
print(f"Tanggal Target Pertama (real_date)   : {first_target_real}")
print(f"Tanggal Target Terakhir (real_date)  : {last_target_real}")

assert n_cutoffs_act == 335, f"Jumlah cutoff {n_cutoffs_act} != 335!"
assert n_targets_act == 344, f"Jumlah target unik {n_targets_act} != 344!"
assert first_cutoff_origin == '2025-04-23', f"First cutoff origin {first_cutoff_origin} != 2025-04-23!"
assert first_target_real == '2025-04-24', f"First target {first_target_real} != 2025-04-24!"
assert n_targets_act == n_cutoffs_act + H - 1, "Formula 335 + 10 - 1 != 344!"
print("✅ Audit CV_Predictions_V4 tervalidasi 100% konsisten.")

# 2. Hitung & Validasi Baseline Evaluasi (v5_helpers)
grp_h1 = df_cv4.groupby('cutoff').head(1)
grp_h10 = df_cv4.groupby('cutoff').tail(1)

hexp_h1, roll_h1, persist_h1 = v5_helpers.compute_baselines(grp_h1, hist_series, cutoff_origins)
hexp_h10, roll_h10, persist_h10 = v5_helpers.compute_baselines(grp_h10, hist_series, cutoff_origins)

mse_hexp_1 = mean_squared_error(grp_h1['y'], hexp_h1)
mse_hexp_10 = mean_squared_error(grp_h10['y'], hexp_h10)
mse_roll_1 = mean_squared_error(grp_h1['y'], roll_h1)
mse_roll_10 = mean_squared_error(grp_h10['y'], roll_h10)

print(f"\nBaseline T0 h=1  (DL-base) | Hist_Exp MSE: {mse_hexp_1:.6e} (Target: 2.689570e-06) | Roll_20 MSE: {mse_roll_1:.6e} (Target: 2.407078e-06)")
print(f"Baseline T0 h=10 (DL-base) | Hist_Exp MSE: {mse_hexp_10:.6e} (Target: 2.686277e-06) | Roll_20 MSE: {mse_roll_10:.6e} (Target: 2.555697e-06)")

assert abs(mse_hexp_1 - 2.689570e-06) < 1e-10, "MSE Hist_Exp h=1 DL-base tidak cocok!"
assert abs(mse_hexp_10 - 2.686277e-06) < 1e-10, "MSE Hist_Exp h=10 DL-base tidak cocok!"
assert abs(mse_roll_1 - 2.407078e-06) < 1e-10, "MSE Roll_20 h=1 tidak cocok!"
assert abs(mse_roll_10 - 2.555697e-06) < 1e-10, "MSE Roll_20 h=10 tidak cocok!"
print("✅ Seluruh baseline Hist_Exp & Roll_20 (DL-base) tervalidasi 100% identik.")

# 3. Benchmark HAR v2 & Dokumentasi Eksplisit Dua Seri Hist_Exp (Poin 4)
har_h1_file = os.path.join(DATASET_DIR, 'har_preds_T0_h1_v2.csv')
har_h10_file = os.path.join(DATASET_DIR, 'har_preds_T0_h10_v2.csv')
assert os.path.exists(har_h1_file), f"File {har_h1_file} tidak ditemukan!"
assert os.path.exists(har_h10_file), f"File {har_h10_file} tidak ditemukan!"

df_har1 = pd.read_csv(har_h1_file)
df_har10 = pd.read_csv(har_h10_file)

mse_har_1 = mean_squared_error(df_har1['True'], df_har1['HAR'])
mse_har_10 = mean_squared_error(df_har10['True'], df_har10['HAR'])
mse_hexp_har_1 = mean_squared_error(df_har1['True'], df_har1['Hist_Exp'])
mse_hexp_har_10 = mean_squared_error(df_har10['True'], df_har10['Hist_Exp'])

print(f"\nHAR v2 T0 h=1  MSE: {mse_har_1:.7e} (Target: 2.2346806e-06)")
print(f"HAR v2 T0 h=10 MSE: {mse_har_10:.7e} (Target: 2.4397431e-06)")
print(f"Hist_Exp (HAR-file) h=1  MSE: {mse_hexp_har_1:.6e} (Target: 2.689342e-06)")
print(f"Hist_Exp (HAR-file) h=10 MSE: {mse_hexp_har_10:.6e} (Target: 2.686052e-06)")

assert abs(mse_har_1 - 2.2346806e-06) < 1e-8, "MSE HAR h=1 tidak cocok!"
assert abs(mse_har_10 - 2.4397431e-06) < 1e-8, "MSE HAR h=10 tidak cocok!"
assert abs(mse_hexp_har_1 - 2.689342e-06) < 1e-8, "MSE Hist_Exp h=1 HAR-file tidak cocok!"
assert abs(mse_hexp_har_10 - 2.686052e-06) < 1e-8, "MSE Hist_Exp h=10 HAR-file tidak cocok!"

USE_HLN = True
# DM T0 h=1: HAR vs Hist_Exp (HAR-file)
res_dm_h1 = v5_helpers.diebold_mariano_test(df_har1['True'], df_har1['HAR'], df_har1['Hist_Exp'], h=1, m_family=8, use_hln=USE_HLN)
print(f"DM T0 h=1 HAR vs Hist_Exp (HAR-file) | stat: {res_dm_h1['stat']:.4f} | p_raw: {res_dm_h1['p_raw']:.4f}")
assert abs(res_dm_h1['stat'] - (-3.0651)) <= 1e-3, "DM T0 h=1 stat HAR vs Hist_Exp mismatch!"
assert abs(res_dm_h1['p_raw'] - 0.0022) <= 1e-4, "DM T0 h=1 p_raw HAR vs Hist_Exp mismatch!"

# DM T0 h=1: HAR vs Roll_20
res_dm_roll1 = v5_helpers.diebold_mariano_test(df_har1['True'], df_har1['HAR'], df_har1['Roll_20'], h=1, m_family=8, use_hln=USE_HLN)
print(f"DM T0 h=1 HAR vs Roll_20             | stat: {res_dm_roll1['stat']:.4f} | p_raw: {res_dm_roll1['p_raw']:.4f}")
assert abs(res_dm_roll1['stat'] - (-1.4992)) <= 1e-3, "DM T0 h=1 stat HAR vs Roll_20 mismatch!"
assert abs(res_dm_roll1['p_raw'] - 0.1338) <= 1e-4, "DM T0 h=1 p_raw HAR vs Roll_20 mismatch!"

# DM T0 h=10: HAR vs Hist_Exp (HAR-file)
res_dm_h10 = v5_helpers.diebold_mariano_test(df_har10['True'], df_har10['HAR'], df_har10['Hist_Exp'], h=10, m_family=8, use_hln=USE_HLN)
print(f"DM T0 h=10 HAR vs Hist_Exp (HAR-file) | stat: {res_dm_h10['stat']:.4f} | p_raw: {res_dm_h10['p_raw']:.4f}")
assert abs(res_dm_h10['stat'] - (-2.6871)) <= 1e-3, "DM T0 h=10 stat HAR vs Hist_Exp mismatch!"
assert abs(res_dm_h10['p_raw'] - 0.0072) <= 1e-4, "DM T0 h=10 p_raw HAR vs Hist_Exp mismatch!"

# DM T0 h=10: HAR vs Roll_20
res_dm_roll10 = v5_helpers.diebold_mariano_test(df_har10['True'], df_har10['HAR'], df_har10['Roll_20'], h=10, m_family=8, use_hln=USE_HLN)
print(f"DM T0 h=10 HAR vs Roll_20            | stat: {res_dm_roll10['stat']:.4f} | p_raw: {res_dm_roll10['p_raw']:.4f}")
assert abs(res_dm_roll10['stat'] - (-0.9228)) <= 1e-3, "DM T0 h=10 stat HAR vs Roll_20 mismatch!"
assert abs(res_dm_roll10['p_raw'] - 0.3561) <= 1e-4, "DM T0 h=10 p_raw HAR vs Roll_20 mismatch!"

# 4. Validasi GARCH Direct Merge
garch_file = os.path.join(DATASET_DIR, 'GARCH_Predictions_V4.csv')
assert os.path.exists(garch_file), f"File {garch_file} tidak ditemukan!"
df_garch = pd.read_csv(garch_file)
m_garch = v5_helpers.load_garch_merged(df_cv4, df_garch)
assert len(m_garch) == 3350, f"Panjang m_garch {len(m_garch)} != 3350!"
print("✅ GARCH direct merge tervalidasi PASS (3350 baris).")

print("\n=== HASIL STAGE A: AUDIT DATA & REPRODUKSI BENCHMARK LULUS 100% ===")
if STAGE == "A":
    print("STAGE='A' selesai. Eksekusi dihentikan sesuai konfigurasi tanpa pelatihan model GPU.")


=== STAGE A: AUDIT PROVENANCE & CROSS-CHECK BENCHMARK (CPU) ===
Duplikat [cutoff, real_date] di CV_Predictions_V4: 0 (PASS: harus 0)

--- AUDIT CV_PREDICTIONS_V4.CSV (Poin 1) ---
Jumlah Cutoff Rolling                : 335 (Harapan: 335)
Jumlah Tanggal Target Unik           : 344 (Harapan: 344)
Tanggal Cutoff Pertama (ds sintetis) : 2023-08-24
Tanggal Cutoff Terakhir (ds sintetis): 2024-07-23
Tanggal Cutoff Pertama (real_date)   : 2025-04-23
Tanggal Cutoff Terakhir (real_date)  : 2026-08-21
Tanggal Target Pertama (real_date)   : 2025-04-24
Tanggal Target Terakhir (real_date)  : 2026-09-04
✅ Audit CV_Predictions_V4 tervalidasi 100% konsisten.

Baseline T0 h=1  (DL-base) | Hist_Exp MSE: 2.689570e-06 (Target: 2.689570e-06) | Roll_20 MSE: 2.407078e-06 (Target: 2.407078e-06)
Baseline T0 h=10 (DL-base) | Hist_Exp MSE: 2.686277e-06 (Target: 2.686277e-06) | Roll_20 MSE: 2.555697e-06 (Target: 2.555697e-06)
✅ Seluruh baseline Hist_Exp & Roll_20 (DL-base) tervalidasi 100% identik.

HAR v2 T0 h=1  

# CELL 5: ARSITEKTUR MODEL FACTORY, RESTORE BEST WEIGHTS CALLBACK, & CHECKSUM


In [5]:
# ======================================================================================
# CELL 5: ARSITEKTUR MODEL FACTORY, RESTORE BEST WEIGHTS CALLBACK, & CHECKSUM
# ======================================================================================
from pytorch_lightning.callbacks import EarlyStopping

def get_weights_checksum_from_dict(state_dict) -> str:
    h = hashlib.md5()
    for k, v in sorted(state_dict.items()):
        h.update(k.encode('utf-8'))
        h.update(v.detach().cpu().numpy().tobytes())
    return h.hexdigest()

def get_weights_checksum(model) -> str:
    return get_weights_checksum_from_dict(model.state_dict())

class RestoreBestWeightsCallback(pl.Callback):
    """
    Callback memori in-memory untuk memulihkan bobot terbaik (restore best weights):
    - Mengabaikan evaluasi validasi sanity-check atau step 0 (agar bobot acak tidak pernah jadi best).
    - Menyimpan klon state_dict di RAM CPU saat validation loss mencapai nilai terendah.
    - Pada on_train_end, memuat kembali bobot terbaik ke pl_module.
    - Menghitung checksum bobot di best step dan membuktikan bobot akhir identik.
    - Merekam actual_final_step, best_step, best_score, stopped_epoch, es_triggered.
    """
    def __init__(self):
        super().__init__()
        self.best_score = None
        self.best_step = None
        self.best_state_dict = None
        self.best_checksum = None
        self.actual_final_step = 0
        self.stopped_epoch = 0
        self.es_triggered = False
        self.restored = False
        self.history = []

    def on_train_batch_end(self, trainer, pl_module, outputs, batch, batch_idx):
        self.actual_final_step = trainer.global_step

    def on_validation_end(self, trainer, pl_module):
        # Abaikan sanity check dan step 0 (bobot acak belum dilatih)
        if getattr(trainer, 'sanity_checking', False) or trainer.global_step == 0:
            return
            
        val_loss = trainer.callback_metrics.get('ptl/val_loss', None)
        if val_loss is not None:
            v_val = float(val_loss)
            self.history.append((trainer.global_step, v_val))
            if self.best_score is None or v_val < self.best_score:
                self.best_score = v_val
                self.best_step = trainer.global_step
                self.best_state_dict = {k: v.detach().cpu().clone() for k, v in pl_module.state_dict().items()}
                self.best_checksum = get_weights_checksum_from_dict(self.best_state_dict)

    def on_train_end(self, trainer, pl_module):
        self.actual_final_step = trainer.global_step
        for cb in trainer.callbacks:
            if isinstance(cb, EarlyStopping):
                self.stopped_epoch = cb.stopped_epoch
                self.es_triggered = (cb.stopped_epoch > 0)

        # Pulihkan bobot terbaik jika tersedia
        if self.best_state_dict is not None:
            pl_module.load_state_dict(self.best_state_dict)
            self.restored = True

def get_model_instance(model_name: str,
                       fset_cols: list,
                       h: int = 10,
                       input_size: int = 50,
                       max_steps: int = 1000,
                       val_check_steps: int = 25,
                       patience: int = 5,
                       seed: int = 42,
                       loss_name: str = 'MAE',
                       extra_kwargs: dict = None):
    """
    Model factory function untuk instansiasi TFT dan NBEATSx secara reproducible.
    Mendukung seleksi loss (MAE vs MSE) dan menyuntikkan RestoreBestWeightsCallback.
    Callback tambahan SELALU di-append/extend, TIDAK PERNAH menimpa EarlyStopping bawaan.
    """
    tracker = RestoreBestWeightsCallback()
    loss_inst = MAE() if loss_name == 'MAE' else MSE()
    
    base_kwargs = {
        'h': h,
        'input_size': input_size,
        'hist_exog_list': fset_cols,
        'max_steps': max_steps,
        'val_check_steps': val_check_steps,
        'early_stop_patience_steps': patience,
        'loss': loss_inst,
        'random_seed': seed,
        'scaler_type': 'standard'
    }
    if extra_kwargs:
        base_kwargs.update(extra_kwargs)

    if model_name == 'TFT':
        model = TFT(**base_kwargs)
    elif model_name == 'NBEATSx':
        model = NBEATSx(**base_kwargs)
    else:
        raise ValueError(f"Model {model_name} tidak didukung!")

    # Konfigurasi Trainer
    model.trainer_kwargs.update({
        'enable_progress_bar': False,
        'enable_checkpointing': False,
        'logger': False
    })
    if torch.cuda.is_available():
        model.trainer_kwargs['accelerator'] = 'cuda'
        model.trainer_kwargs['devices'] = 1

    # PENTING: Append callback tracker ke daftar yang sudah ada (JANGAN TIMPA!)
    if 'callbacks' not in model.trainer_kwargs or model.trainer_kwargs['callbacks'] is None:
        model.trainer_kwargs['callbacks'] = []
    model.trainer_kwargs['callbacks'].append(tracker)

    # Assert bahwa EarlyStopping bawaan NF tetap ada
    has_es = any(isinstance(cb, EarlyStopping) for cb in model.trainer_kwargs['callbacks'])
    assert has_es, "ASSERT STOP: EarlyStopping bawaan hilang dari trainer_kwargs['callbacks']!"

    return model

print("✅ Model Factory & RestoreBestWeightsCallback callback siap.")


✅ Model Factory & RestoreBestWeightsCallback callback siap.


# CELL 6: STAGE B - TUNING RINGAN HYPERPARAMETER (TRAIN 1165 + VAL 167 = 1332 SAJA, NO TEST SET)


In [6]:
# ======================================================================================
# CELL 6: STAGE B - TUNING RINGAN HYPERPARAMETER (TRAIN 1165 + VAL 167 = 1332 SAJA, NO TEST SET)
# ======================================================================================

STAGE_B_SEEDS = [42, 7]

TUNING_GRIDS = {
    'TFT': [
        {'input_size': 20, 'learning_rate': 1e-3, 'hidden_size': 64, 'loss_name': 'MAE'},
        {'input_size': 20, 'learning_rate': 1e-3, 'hidden_size': 64, 'loss_name': 'MSE'},
        {'input_size': 20, 'learning_rate': 3e-4, 'hidden_size': 64, 'loss_name': 'MAE'},
        {'input_size': 20, 'learning_rate': 3e-4, 'hidden_size': 64, 'loss_name': 'MSE'},
        {'input_size': 50, 'learning_rate': 1e-3, 'hidden_size': 64, 'loss_name': 'MAE'},
        {'input_size': 50, 'learning_rate': 1e-3, 'hidden_size': 64, 'loss_name': 'MSE'},
        {'input_size': 50, 'learning_rate': 3e-4, 'hidden_size': 64, 'loss_name': 'MAE'},
        {'input_size': 50, 'learning_rate': 3e-4, 'hidden_size': 64, 'loss_name': 'MSE'},
    ],
    'NBEATSx': [
        {'input_size': 20, 'learning_rate': 1e-3, 'mlp_units': [[256, 256], [256, 256], [256, 256]], 'loss_name': 'MAE'},
        {'input_size': 20, 'learning_rate': 1e-3, 'mlp_units': [[256, 256], [256, 256], [256, 256]], 'loss_name': 'MSE'},
        {'input_size': 20, 'learning_rate': 3e-4, 'mlp_units': [[256, 256], [256, 256], [256, 256]], 'loss_name': 'MAE'},
        {'input_size': 20, 'learning_rate': 3e-4, 'mlp_units': [[256, 256], [256, 256], [256, 256]], 'loss_name': 'MSE'},
        {'input_size': 50, 'learning_rate': 1e-3, 'mlp_units': [[256, 256], [256, 256], [256, 256]], 'loss_name': 'MAE'},
        {'input_size': 50, 'learning_rate': 1e-3, 'mlp_units': [[256, 256], [256, 256], [256, 256]], 'loss_name': 'MSE'},
        {'input_size': 50, 'learning_rate': 3e-4, 'mlp_units': [[256, 256], [256, 256], [256, 256]], 'loss_name': 'MAE'},
        {'input_size': 50, 'learning_rate': 3e-4, 'mlp_units': [[256, 256], [256, 256], [256, 256]], 'loss_name': 'MSE'},
    ]
}

tuning_log_file = os.path.join(OUTPUT_DIR, "v6_tuning_log.csv")
best_hp_file = os.path.join(OUTPUT_DIR, "v6_best_hyperparameters.json")

if STAGE in ["B", "ALL"]:
    print("=== STAGE B: TUNING RINGAN HYPERPARAMETER (TRAIN 1165 + VAL 167 = 1332 SAJA) ===")
    print(f"Model: TFT & NBEATSx | Grid: 8 per model | Seeds: {STAGE_B_SEEDS} | FeatureSet: NOSPILL (11 fitur)")
    print(f"Data: Train={len(df_train)} (1165) | Val={len(df_val)} (167) | Total Fit={len(df_train_val)} (1332)")
    print(f"Test Set (344 baris target, 335 cutoffs): DILARANG/TIDAK DISENTUH SAMA SEKALI")
    print("Catatan Metodologis: Dimensi loss (MAE vs MSE) ditetapkan SEBELUM melihat hasil test.")
    
    assert len(df_train_val) == 1332, f"ASSERT STOP: Data fit Stage B harus 1332 baris, ditemukan {len(df_train_val)}!"
    
    tuning_records = []
    
    for m_name in ['TFT', 'NBEATSx']:
        grid = TUNING_GRIDS[m_name]
        fset_cols = FEATURE_SETS['NOSPILL']
        
        for cfg_idx, hp_cfg in enumerate(grid):
            cfg_id = f"{m_name}_cfg{cfg_idx+1}"
            loss_choice = hp_cfg['loss_name']
            extra_kw = {k: v for k, v in hp_cfg.items() if k not in ['input_size', 'loss_name']}
            seed_scores = []
            
            for s in STAGE_B_SEEDS:
                set_all_seeds(s)
                
                m = get_model_instance(
                    model_name=m_name,
                    fset_cols=fset_cols,
                    h=H,
                    input_size=hp_cfg['input_size'],
                    max_steps=MAX_STEPS,
                    val_check_steps=VAL_CHECK_STEPS,
                    patience=CHOSEN_PATIENCE,
                    seed=s,
                    loss_name=loss_choice,
                    extra_kwargs=extra_kw
                )
                
                nf_sub = df_train_val[['unique_id', 'ds', 'y'] + fset_cols].copy()
                nf_engine = NeuralForecast(models=[m], freq='D')
                
                # Fit di train (1165), validasi di val (167)
                t_start = time.time()
                nf_engine.fit(df=nf_sub, val_size=n_val)
                t_fit = time.time() - t_start
                
                m_fitted = nf_engine.models[0]
                chk_before = get_weights_checksum(m_fitted)
                
                tr_list = [cb for cb in m_fitted.trainer_kwargs.get('callbacks', []) if isinstance(cb, RestoreBestWeightsCallback)]
                tr = tr_list[0] if tr_list else None
                act_step = tr.actual_final_step if tr else getattr(m_fitted, 'global_step', 0)
                
                if tr and tr.best_checksum:
                    assert chk_before == tr.best_checksum, f"ASSERT STOP: Bobot {cfg_id} s{s} tidak sama dengan best step!"
                    assert tr.best_step > 0, f"ASSERT STOP: Best step {tr.best_step} harus > 0 (sanity check bocor)!"
                
                # Evaluasi rolling pada validation window (158 cutoffs dengan h=10 berakhir di baris 1331)
                n_val_windows = n_val - H + 1  # 167 - 10 + 1 = 158
                cv_val = nf_engine.cross_validation(
                    df=nf_sub,
                    n_windows=n_val_windows,
                    step_size=1,
                    use_fitted=True
                ).reset_index(drop=True)
                
                chk_after = get_weights_checksum(m_fitted)
                assert chk_before == chk_after, f"ASSERT STOP: Bobot {cfg_id} s{s} berubah saat cross_validation!"
                
                # Evaluasi MSE pada Val untuk h=1 dan h=10
                grp_v1 = cv_val.groupby('cutoff').head(1)
                grp_v10 = cv_val.groupby('cutoff').tail(1)
                mse_v1 = float(mean_squared_error(grp_v1['y'], grp_v1[m_name]))
                mse_v10 = float(mean_squared_error(grp_v10['y'], grp_v10[m_name]))
                mse_v_avg = (mse_v1 + mse_v10) / 2.0
                
                record = {
                    'model': m_name,
                    'config_id': cfg_id,
                    'seed': s,
                    'loss_train': loss_choice,
                    'hyperparameters': json.dumps(hp_cfg),
                    'actual_final_step': act_step,
                    'best_step': tr.best_step if tr else None,
                    'es_triggered': tr.es_triggered if tr else False,
                    'weights_restored': tr.restored if tr else False,
                    'fit_seconds': round(t_fit, 2),
                    'val_mse_h1': mse_v1,
                    'val_mse_h10': mse_v10,
                    'val_mse_avg': mse_v_avg,
                    'weights_checksum': chk_before
                }
                tuning_records.append(record)
                seed_scores.append(mse_v_avg)
                print(f"  [{cfg_id}] Loss: {loss_choice:<3} | Seed {s:<2} | Final: {act_step:<4} | Best: {tr.best_step:<4} | Val MSE Avg: {mse_v_avg:.6e}")
                
    df_tuning = pd.DataFrame(tuning_records)
    df_tuning.to_csv(tuning_log_file, index=False)
    print(f"\n✅ Log tuning lengkap tersimpan ke: {tuning_log_file}")
    
    best_hp = {}
    for m_name in ['TFT', 'NBEATSx']:
        df_m = df_tuning[df_tuning['model'] == m_name]
        agg_cfg = df_m.groupby(['config_id', 'hyperparameters'])[['val_mse_h1', 'val_mse_h10', 'val_mse_avg']].mean().reset_index()
        agg_cfg = agg_cfg.sort_values('val_mse_avg').reset_index(drop=True)
        winner = agg_cfg.iloc[0]
        best_hp[m_name] = json.loads(winner['hyperparameters'])
        print(f"\n🏆 Pemenang Tuning {m_name}: {winner['config_id']} | Avg Val MSE: {winner['val_mse_avg']:.6e}")
        print(f"   Hyperparameters: {best_hp[m_name]}")
        
    with open(best_hp_file, 'w') as f_out:
        json.dump(best_hp, f_out, indent=2)
    print(f"✅ Hyperparameter terpilih disimpan ke: {best_hp_file}")

else:
    if os.path.exists(best_hp_file):
        with open(best_hp_file, 'r') as f_in:
            best_hp = json.load(f_in)
        print(f"Stage B dilewati. Memuat hyperparameter terbaik dari: {best_hp_file}")
    else:
        best_hp = {
            'TFT': {'input_size': 50, 'learning_rate': 1e-3, 'hidden_size': 64, 'loss_name': 'MAE'},
            'NBEATSx': {'input_size': 50, 'learning_rate': 1e-3, 'mlp_units': [[256, 256], [256, 256], [256, 256]], 'loss_name': 'MAE'}
        }
        print(f"Stage B dilewati dan file tidak ditemukan. Menggunakan konfigurasi dasar a priori: {best_hp}")


Seed set to 42
Seed set to 42


=== STAGE B: TUNING RINGAN HYPERPARAMETER (TRAIN 1165 + VAL 167 = 1332 SAJA) ===
Model: TFT & NBEATSx | Grid: 8 per model | Seeds: [42, 7] | FeatureSet: NOSPILL (11 fitur)
Data: Train=1165 (1165) | Val=167 (167) | Total Fit=1332 (1332)
Test Set (344 baris target, 335 cutoffs): DILARANG/TIDAK DISENTUH SAMA SEKALI
Catatan Metodologis: Dimensi loss (MAE vs MSE) ditetapkan SEBELUM melihat hasil test.


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


┏━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name                    ┃ Type                     ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ loss                    │ MAE                      │      0 │ train │     0 │
│ 1 │ hist_cat_embeddings     │ ModuleList               │      0 │ train │     0 │
│ 2 │ futr_cat_embeddings     │ ModuleList               │      0 │ train │     0 │
│ 3 │ stat_cat_embeddings     │ ModuleList               │      0 │ train │     0 │
│ 4 │ padder_train            │ ConstantPad1d            │      0 │ train │     0 │
│ 5 │ scaler                  │ TemporalNorm             │      0 │ train │     0 │
│ 6 │ embedding               │ TFTEmbedding             │  1.7 K │ train │     0 │
│ 7 │ temporal_encoder        │ TemporalCovariateEncoder │  396 K │ train │     0 │
│ 8 │ temporal_fusion_decoder │ TemporalFusionDecoder    │ 64.8 K │ train │     0 │
│ 9 │ output_adapter          │ Linear                   │     65 │ train │     0 │
└───┴─────────────────────────┴──────────────────────────┴────────┴───────┴───────┘

Trainable params: 462 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 462 K                                                                                                
Total estimated model params size (MB): 1.852                                                                      
Modules in train mode: 179                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Trainer already configured with model summary callbacks: [<class 'pytorch_lightning.callbacks.rich_model_summary.RichModelSummary'>]. Skipping setting a default `ModelSummary` callback.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
Seed set to 7
Seed set to 7
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


... (121 outputs left out)

┏━━━┳━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name                ┃ Type          ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ loss                │ MSE           │      0 │ train │     0 │
│ 1 │ hist_cat_embeddings │ ModuleList    │      0 │ train │     0 │
│ 2 │ futr_cat_embeddings │ ModuleList    │      0 │ train │     0 │
│ 3 │ stat_cat_embeddings │ ModuleList    │      0 │ train │     0 │
│ 4 │ padder_train        │ ConstantPad1d │      0 │ train │     0 │
│ 5 │ scaler              │ TemporalNorm  │      0 │ train │     0 │
│ 6 │ blocks              │ ModuleList    │  1.1 M │ train │     0 │
└───┴─────────────────────┴───────────────┴────────┴───────┴───────┘

Trainable params: 1.1 M                                                                                            
Non-trainable params: 1.3 K                                                                                        
Total params: 1.1 M                                                                                                
Total estimated model params size (MB): 4.325                                                                      
Modules in train mode: 34                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Trainer already configured with model summary callbacks: [<class 'pytorch_lightning.callbacks.rich_model_summary.RichModelSummary'>]. Skipping setting a default `ModelSummary` callback.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


  [NBEATSx_cfg8] Loss: MSE | Seed 7  | Final: 150  | Best: 25   | Val MSE Avg: 2.737569e-06

✅ Log tuning lengkap tersimpan ke: ./v6_outputs/v6_tuning_log.csv

🏆 Pemenang Tuning TFT: TFT_cfg6 | Avg Val MSE: 1.562859e-06
   Hyperparameters: {'input_size': 50, 'learning_rate': 0.001, 'hidden_size': 64, 'loss_name': 'MSE'}

🏆 Pemenang Tuning NBEATSx: NBEATSx_cfg3 | Avg Val MSE: 1.772739e-06
   Hyperparameters: {'input_size': 20, 'learning_rate': 0.0003, 'mlp_units': [[256, 256], [256, 256], [256, 256]], 'loss_name': 'MAE'}
✅ Hyperparameter terpilih disimpan ke: ./v6_outputs/v6_best_hyperparameters.json


# CELL 7: STAGE C - MULTI-SEED MAIN RUN (5 SEEDS x 4 CONFIGS, 335 TEST CUTOFFS, KOREKSI FIT 1332)


In [7]:
# ======================================================================================
# CELL 7: STAGE C - MULTI-SEED MAIN RUN (5 SEEDS x 4 CONFIGS, 335 TEST CUTOFFS, KOREKSI FIT 1332)
# ======================================================================================

es_log_file = os.path.join(OUTPUT_DIR, "v6_es_log.csv")

if STAGE in ["C", "ALL"]:
    print("=== STAGE C: RUN UTAMA MULTI-SEED (335 CUTOFF ROLLING TEST) ===")
    print(f"Seeds: {SEEDS} | Configs: {CONFIGS} | Total Run: {len(SEEDS) * len(CONFIGS)}")
    print(f"Hyperparameters Terpilih (dari Stage B):")
    for m_k, hp_v in best_hp.items():
        print(f"  - {m_k}: {hp_v}")

    # ==================================================================================
    # ASSERT ANTI-LEAKAGE KETAT (Poin 1 & 3)
    # ==================================================================================
    # 1. Pastikan df_train_val tepat 1332 baris (baris 1 s.d. 1332)
    assert len(df_train_val) == 1332, f"ASSERT STOP: Data fit harus tepat 1332 baris, ditemukan {len(df_train_val)}!"
    assert len(df_test) == 344, f"ASSERT STOP: Data test target harus tepat 344 baris, ditemukan {len(df_test)}!"
    
    max_fit_real_date = pd.to_datetime(df_train_val['real_date'].max())
    min_test_real_date = pd.to_datetime(df_test['real_date'].min())
    max_fit_ds = pd.to_datetime(df_train_val['ds'].max())
    min_test_ds = pd.to_datetime(df_test['ds'].min())
    
    print("\n--- VALIDASI KETAT ANTI-LEAKAGE (Poin 1) ---")
    print(f"Jumlah baris data fit (Train + Val)      : {len(df_train_val)} (baris 1 s.d. 1332)")
    print(f"Jumlah baris target test set             : {len(df_test)} (baris 1333 s.d. 1676, 335 cutoffs)")
    print(f"Tanggal Maksimal Data Fit (real_date)   : {max_fit_real_date.strftime('%Y-%m-%d')}")
    print(f"Tanggal Cutoff Pertama CV4 (real_date)  : {first_cutoff_origin}")
    print(f"Tanggal Target Pertama Window 1 (real)  : {min_test_real_date.strftime('%Y-%m-%d')}")
    print(f"Tanggal Maksimal Data Fit (ds sintetis) : {max_fit_ds.strftime('%Y-%m-%d')}")
    print(f"Tanggal Target Pertama Window 1 (ds)    : {min_test_ds.strftime('%Y-%m-%d')}")
    
    # Assert (a): max real_date data fit == tanggal cutoff pertama
    assert max_fit_real_date == pd.to_datetime(first_cutoff_origin), f"ASSERT (a) GAGAL: max_fit_real_date ({max_fit_real_date}) != first_cutoff_origin ({first_cutoff_origin})"
    # Assert (b): tanggal target pertama window pertama > max real_date data fit
    assert min_test_real_date > max_fit_real_date, f"ASSERT (b) GAGAL: min_test_real_date ({min_test_real_date}) <= max_fit_real_date ({max_fit_real_date})"
    print("✅ Assert (a) & (b) PASS: Data fit berakhir persis di tanggal cutoff pertama dan sebelum target window pertama!")

    # 1. Smoke Test 335 Cutoff & Estimasi Waktu GPU
    print("\n--- Smoke Test 335 Cutoff & Assert (c) Anti-Leakage ---")
    smoke_cfg = CONFIGS[0]
    smoke_model_name, smoke_fset_name, _ = smoke_cfg
    smoke_hp = best_hp[smoke_model_name]
    smoke_loss = smoke_hp.get('loss_name', 'MAE')
    smoke_extra = {k: v for k, v in smoke_hp.items() if k not in ['input_size', 'loss_name']}
    
    set_all_seeds(42)
    m_smoke = get_model_instance(
        model_name=smoke_model_name,
        fset_cols=FEATURE_SETS[smoke_fset_name],
        h=H,
        input_size=smoke_hp['input_size'],
        max_steps=20,
        val_check_steps=10,
        patience=2,
        seed=42,
        loss_name=smoke_loss,
        extra_kwargs=smoke_extra
    )
    nf_smoke = NeuralForecast(models=[m_smoke], freq='D')
    nf_sub_smoke = df_train_val[['unique_id', 'ds', 'y'] + FEATURE_SETS[smoke_fset_name]].copy()
    
    t0_sm = time.time()
    nf_smoke.fit(df=nf_sub_smoke, val_size=n_val)
    t_smoke_fit = time.time() - t0_sm
    time_per_step = t_smoke_fit / 20.0
    
    t0_cv_sm = time.time()
    cv_sm = nf_smoke.cross_validation(
        df=nf_df[['unique_id', 'ds', 'y'] + FEATURE_SETS[smoke_fset_name]],
        n_windows=n_test_cutoffs,
        step_size=1,
        use_fitted=True
    )
    t_smoke_cv = time.time() - t0_cv_sm
    
    cutoff_min = pd.to_datetime(cv_sm['cutoff'].min())
    cutoff_max = pd.to_datetime(cv_sm['cutoff'].max())
    cutoff_nunique = cv_sm['cutoff'].nunique()
    ds_min = pd.to_datetime(cv_sm['ds'].min())
    ds_max = pd.to_datetime(cv_sm['ds'].max())
    ds_nunique = cv_sm['ds'].nunique()
    expected_min_ds = max_fit_ds + pd.Timedelta(days=1)
    
    print("\n=== RAW MENTAH OBJEK CV_TEST SMOKE TEST (335 WINDOWS) ===")
    print(f"  - cv_test['cutoff'].min()     : {cutoff_min.strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"  - cv_test['cutoff'].max()     : {cutoff_max.strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"  - cv_test['cutoff'].nunique() : {cutoff_nunique} (Harapan: 335)")
    print(f"  - cv_test['ds'].min()         : {ds_min.strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"  - cv_test['ds'].max()         : {ds_max.strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"  - cv_test['ds'].nunique()     : {ds_nunique} (Harapan: 344)")
    print(f"  - max_fit_ds                  : {max_fit_ds.strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"  - max_fit_ds + 1 hari         : {expected_min_ds.strftime('%Y-%m-%d %H:%M:%S')}")
    
    assert cutoff_nunique == 335, f"ASSERT STOP: Cutoff unik {cutoff_nunique} != 335!"
    assert ds_nunique == 344, f"ASSERT STOP: ds unik {ds_nunique} != 344!"
    assert cutoff_min == max_fit_ds, f"ASSERT STOP: cv_test['cutoff'].min() ({cutoff_min}) != max_fit_ds ({max_fit_ds})!"
    assert ds_min == expected_min_ds, f"ASSERT STOP: cv_test['ds'].min() ({ds_min}) != max_fit_ds + 1 hari ({expected_min_ds})!"
    print("✅ Assert (c) PASS: cv_test['cutoff'].min() == max_ds_fit (2023-08-24) dan cv_test['ds'].min() == max_ds_fit + 1 hari (2023-08-25)!")
    
    # Estimasi per konfigurasi-seed (1000 step + 335 cutoff CV)
    est_sec_fit = time_per_step * MAX_STEPS
    est_sec_cv = t_smoke_cv
    est_sec_per_cfg_seed = est_sec_fit + est_sec_cv
    total_runs = len(SEEDS) * len(CONFIGS)
    est_total_hours = (est_sec_per_cfg_seed * total_runs) / 3600.0
    
    print(f"\nHasil Smoke Test:")
    print(f"  - Waktu Fit (20 steps)       : {t_smoke_fit:.2f} s ({time_per_step*1000:.2f} ms/step)")
    print(f"  - Waktu CV (335 cutoffs)     : {t_smoke_cv:.2f} s ({t_smoke_cv/n_test_cutoffs*1000:.2f} ms/cutoff)")
    print(f"  - Estimasi per (config, seed): ~{est_sec_per_cfg_seed/60:.2f} menit")
    print(f"  - Estimasi Total ({total_runs} runs): ~{est_total_hours:.2f} jam")
    
    if est_total_hours > 9.0:
        msg = (
            f"\n🚨 PERINGATAN KERAS: Estimasi total waktu eksekusi ({est_total_hours:.2f} jam) "
            f"MELEBIHI batas 9 jam Kaggle!\n"
            f"Saran Pembagian Sesi Kaggle:\n"
            f"  - Sesi 1: Jalankan dengan SEEDS = [42, 7]\n"
            f"  - Sesi 2: Pasang output Sesi 1, jalankan dengan SEEDS = [13, 21]\n"
            f"  - Sesi 3: Pasang output Sesi 1 & 2, jalankan dengan SEEDS = [99] dan STAGE = 'D'\n"
            f"Sistem menghentikan eksekusi sekarang agar tidak terputus paksa di tengah jalan."
        )
        print(msg)
        raise RuntimeError(msg)
    else:
        print(f"✅ Estimasi waktu aman ({est_total_hours:.2f} jam <= 9 jam). Melanjutkan eksekusi utama...")
        
    # 2. Main Execution Loop
    es_records = []
    if os.path.exists(es_log_file):
        try:
            es_records = pd.read_csv(es_log_file).to_dict('records')
        except Exception:
            es_records = []
            
    for seed in SEEDS:
        for (m_name, fset_name, target_label) in CONFIGS:
            fset_cols = FEATURE_SETS[fset_name]
            hp = best_hp[m_name]
            input_size = hp['input_size']
            loss_choice = hp.get('loss_name', 'MAE')
            extra_kw = {k: v for k, v in hp.items() if k not in ['input_size', 'loss_name']}
            
            partial_file = os.path.join(OUTPUT_DIR, f"v6_partial_{m_name}_{fset_name}_T0_s{seed}.csv")
            
            if os.path.exists(partial_file):
                print(f"  [RESUME] Menemukan checkpoint parsial {partial_file}. Melewati training...")
                continue
                
            print(f"\n--> Melatih {m_name} | FeatureSet: {fset_name} | Seed: {seed} | Loss: {loss_choice} (335 cutoff)...")
            set_all_seeds(seed)
            
            m = get_model_instance(
                model_name=m_name,
                fset_cols=fset_cols,
                h=H,
                input_size=input_size,
                max_steps=MAX_STEPS,
                val_check_steps=VAL_CHECK_STEPS,
                patience=CHOSEN_PATIENCE,
                seed=seed,
                loss_name=loss_choice,
                extra_kwargs=extra_kw
            )
            
            nf_sub = nf_df[['unique_id', 'ds', 'y'] + fset_cols].copy()
            df_train_val_sub = df_train_val[['unique_id', 'ds', 'y'] + fset_cols].copy()
            
            nf_engine = NeuralForecast(models=[m], freq='D')
            
            # FIT TEPAT SATU KALI pada baris 1-1332 (1165 train, 167 val)
            t_fit_start = time.time()
            nf_engine.fit(df=df_train_val_sub, val_size=n_val)
            fit_time = time.time() - t_fit_start
            
            m_trained = nf_engine.models[0]
            chk_before = get_weights_checksum(m_trained)
            
            tr_list = [cb for cb in m_trained.trainer_kwargs.get('callbacks', []) if isinstance(cb, RestoreBestWeightsCallback)]
            tr = tr_list[0] if tr_list else None
            actual_step = tr.actual_final_step if tr else 0
            best_st = tr.best_step if tr else None
            best_sc = tr.best_score if tr else None
            stopped_ep = tr.stopped_epoch if tr else 0
            es_trig = tr.es_triggered if tr else False
            restored = tr.restored if tr else False
            
            if tr and tr.best_checksum:
                assert chk_before == tr.best_checksum, f"ASSERT STOP: Bobot {m_name}_{fset_name}_s{seed} tidak sesuai dengan best step!"
                assert tr.best_step > 0, f"ASSERT STOP: Best step {tr.best_step} harus > 0!"
            
            print(f"      Fit selesai dalam {fit_time:.2f} s | Final Step: {actual_step} | Best Step: {best_st} | ES: {es_trig} | Restored: {restored}")
            
            es_records.append({
                'model': m_name,
                'feature_set': fset_name,
                'target': target_label,
                'seed': seed,
                'loss_train': loss_choice,
                'max_steps': MAX_STEPS,
                'val_check_steps': VAL_CHECK_STEPS,
                'patience_checks': CHOSEN_PATIENCE,
                'actual_final_step': actual_step,
                'best_step': best_st,
                'best_val_loss': best_sc,
                'stopped_epoch': stopped_ep,
                'es_triggered': es_trig,
                'weights_restored': restored,
                'weights_checksum_restored': chk_before,
                'fit_seconds': round(fit_time, 2)
            })
            pd.DataFrame(es_records).to_csv(es_log_file, index=False)
            
            # ROLLING INFERENCE OUT-OF-SAMPLE (335 cutoff)
            t_cv_start = time.time()
            cv_res = nf_engine.cross_validation(
                df=nf_sub,
                n_windows=n_test_cutoffs,
                step_size=1,
                use_fitted=True
            ).reset_index(drop=True)
            cv_time = time.time() - t_cv_start
            
            chk_after = get_weights_checksum(m_trained)
            print(f"      CV selesai dalam {cv_time:.2f} s ({cv_time/n_test_cutoffs:.3f} s/cutoff). Checksum match: {chk_before == chk_after}")
            assert chk_before == chk_after, f"ASSERT STOP: Bobot {m_name}_{fset_name}_s{seed} berubah saat cross_validation!"
            
            cv_res['ds'] = pd.to_datetime(cv_res['ds'])
            nf_dates = nf_df[['unique_id', 'ds', 'real_date']].copy()
            nf_dates['ds'] = pd.to_datetime(nf_dates['ds'])
            cv_res = pd.merge(cv_res, nf_dates, on=['unique_id', 'ds'], how='left')
            
            cv_res.to_csv(partial_file, index=False)
            print(f"      ✅ Checkpoint tersimpan: {partial_file}")
            
            if m_name == 'TFT' and fset_name == 'SPILL':
                try:
                    _ = nf_engine.predict(df=df_train_val_sub)
                    if hasattr(m_trained, 'feature_importances') and callable(getattr(m_trained, 'feature_importances')):
                        imp_dict = m_trained.feature_importances()
                        if isinstance(imp_dict, dict) and 'Past variable importance over time' in imp_dict:
                            df_raw_imp = imp_dict['Past variable importance over time']
                            mean_imp = df_raw_imp.mean().reset_index()
                            mean_imp.columns = ['feature', 'mean_importance']
                            df_imp = mean_imp[~mean_imp['feature'].isin(['unique_id', 'ds', 'y'])].copy()
                            df_imp['seed'] = seed
                            imp_file = os.path.join(OUTPUT_DIR, f"v6_tft_feature_importance_s{seed}.csv")
                            df_imp.to_csv(imp_file, index=False)
                            print(f"      TFT Importance tersimpan: {imp_file}")
                except Exception as e_imp:
                    print(f"      ⚠️ Gagal mengekstrak TFT importance s{seed}: {e_imp}")

    print("\n✅ Seluruh run Stage C selesai dengan sukses.")


Seed set to 42
Seed set to 42


=== STAGE C: RUN UTAMA MULTI-SEED (335 CUTOFF ROLLING TEST) ===
Seeds: [42, 7, 13, 21, 99] | Configs: [('TFT', 'SPILL', 'T0'), ('TFT', 'NOSPILL', 'T0'), ('NBEATSx', 'SPILL', 'T0'), ('NBEATSx', 'NOSPILL', 'T0')] | Total Run: 20
Hyperparameters Terpilih (dari Stage B):
  - TFT: {'input_size': 50, 'learning_rate': 0.001, 'hidden_size': 64, 'loss_name': 'MSE'}
  - NBEATSx: {'input_size': 20, 'learning_rate': 0.0003, 'mlp_units': [[256, 256], [256, 256], [256, 256]], 'loss_name': 'MAE'}

--- VALIDASI KETAT ANTI-LEAKAGE (Poin 1) ---
Jumlah baris data fit (Train + Val)      : 1332 (baris 1 s.d. 1332)
Jumlah baris target test set             : 344 (baris 1333 s.d. 1676, 335 cutoffs)
Tanggal Maksimal Data Fit (real_date)   : 2025-04-23
Tanggal Cutoff Pertama CV4 (real_date)  : 2025-04-23
Tanggal Target Pertama Window 1 (real)  : 2025-04-24
Tanggal Maksimal Data Fit (ds sintetis) : 2023-08-24
Tanggal Target Pertama Window 1 (ds)    : 2023-08-25
✅ Assert (a) & (b) PASS: Data fit berakhir persis d

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


┏━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name                    ┃ Type                     ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ loss                    │ MSE                      │      0 │ train │     0 │
│ 1 │ hist_cat_embeddings     │ ModuleList               │      0 │ train │     0 │
│ 2 │ futr_cat_embeddings     │ ModuleList               │      0 │ train │     0 │
│ 3 │ stat_cat_embeddings     │ ModuleList               │      0 │ train │     0 │
│ 4 │ padder_train            │ ConstantPad1d            │      0 │ train │     0 │
│ 5 │ scaler                  │ TemporalNorm             │      0 │ train │     0 │
│ 6 │ embedding               │ TFTEmbedding             │  2.8 K │ train │     0 │
│ 7 │ temporal_encoder        │ TemporalCovariateEncoder │  605 K │ train │     0 │
│ 8 │ temporal_fusion_decoder │ TemporalFusionDecoder    │ 64.8 K │ train │     0 │
│ 9 │ output_adapter          │ Linear                   │     65 │ train │     0 │
└───┴─────────────────────────┴──────────────────────────┴────────┴───────┴───────┘

Trainable params: 673 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 673 K                                                                                                
Total estimated model params size (MB): 2.693                                                                      
Modules in train mode: 251                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

`Trainer.fit` stopped: `max_steps=20` reached.
Trainer already configured with model summary callbacks: [<class 'pytorch_lightning.callbacks.rich_model_summary.RichModelSummary'>]. Skipping setting a default `ModelSummary` callback.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
Seed set to 42
Seed set to 42
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


... (108 outputs left out)

┏━━━┳━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name                ┃ Type          ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ loss                │ MAE           │      0 │ train │     0 │
│ 1 │ hist_cat_embeddings │ ModuleList    │      0 │ train │     0 │
│ 2 │ futr_cat_embeddings │ ModuleList    │      0 │ train │     0 │
│ 3 │ stat_cat_embeddings │ ModuleList    │      0 │ train │     0 │
│ 4 │ padder_train        │ ConstantPad1d │      0 │ train │     0 │
│ 5 │ scaler              │ TemporalNorm  │      0 │ train │     0 │
│ 6 │ blocks              │ ModuleList    │  796 K │ train │     0 │
└───┴─────────────────────┴───────────────┴────────┴───────┴───────┘

Trainable params: 795 K                                                                                            
Non-trainable params: 630                                                                                          
Total params: 796 K                                                                                                
Total estimated model params size (MB): 3.185                                                                      
Modules in train mode: 34                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Trainer already configured with model summary callbacks: [<class 'pytorch_lightning.callbacks.rich_model_summary.RichModelSummary'>]. Skipping setting a default `ModelSummary` callback.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


      Fit selesai dalam 1.84 s | Final Step: 175 | Best Step: 50 | ES: True | Restored: True
      CV selesai dalam 0.15 s (0.000 s/cutoff). Checksum match: True
      ✅ Checkpoint tersimpan: ./v6_outputs/v6_partial_NBEATSx_NOSPILL_T0_s99.csv

✅ Seluruh run Stage C selesai dengan sukses.


# CELL 8: STAGE D - AGREGASI MULTI-SEED, ENSEMBLE, METRIK & UJI DIEBOLD-MARIANO (CPU)


In [8]:
# ======================================================================================
# CELL 8: STAGE D - AGREGASI MULTI-SEED, ENSEMBLE, METRIK & UJI DIEBOLD-MARIANO (CPU)
# ======================================================================================

if STAGE in ["D", "ALL"]:
    print("=== STAGE D: AGREGASI, ENSEMBLE, & UJI DIEBOLD-MARIANO (CPU) ===")
    
    raw_preds = {}
    for seed in SEEDS:
        for (m_name, fset_name, tgt) in CONFIGS:
            cfg_k = f"{m_name}_{fset_name}_{tgt}_s{seed}"
            p_file = os.path.join(OUTPUT_DIR, f"v6_partial_{m_name}_{fset_name}_{tgt}_s{seed}.csv")
            if not os.path.exists(p_file):
                raise FileNotFoundError(f"File partial tidak ditemukan: {p_file}. Pastikan Stage C sudah selesai!")
            raw_preds[cfg_k] = pd.read_csv(p_file)
            
    print(f"Berhasil memuat {len(raw_preds)} file partial ({len(SEEDS)} seeds x {len(CONFIGS)} configs).")
    
    df_har_preds = {
        'h=1': pd.read_csv(os.path.join(DATASET_DIR, "har_preds_T0_h1_v2.csv")),
        'h=10': pd.read_csv(os.path.join(DATASET_DIR, "har_preds_T0_h10_v2.csv"))
    }
    
    per_seed_metrics = []
    
    for (m_name, fset_name, tgt) in CONFIGS:
        for seed in SEEDS:
            cfg_k = f"{m_name}_{fset_name}_{tgt}_s{seed}"
            df_p = raw_preds[cfg_k]
            c_origins = v5_helpers.get_cutoff_origins(df_p, hist_series)
            
            for h_val, h_step in [(1, 'h=1'), (10, 'h=10')]:
                grp = df_p.groupby('cutoff').head(1) if h_val == 1 else df_p.groupby('cutoff').tail(1)
                yt = grp['y'].values
                yp = grp[m_name].values
                hexp_arr, roll_arr, _ = v5_helpers.compute_baselines(grp, hist_series, c_origins)
                
                m_eval = v5_helpers.evaluate_metrics(yt, yp, hexp_arr, roll_arr)
                
                bias_lvl = float(np.mean(yp) / np.mean(yt)) if np.mean(yt) > 0 else np.nan
                pct_le_0 = float(np.mean(yp <= 0) * 100.0)
                
                per_seed_metrics.append({
                    'Model': m_name,
                    'FeatureSet': fset_name,
                    'Target': tgt,
                    'Seed': seed,
                    'Horizon': h_step,
                    'N': m_eval['N'],
                    'MSE': m_eval['MSE'],
                    'MAE': m_eval['MAE'],
                    'RMSE': m_eval['RMSE'],
                    'R2_OOS_HistExp': m_eval['R2_OOS_HistExp'],
                    'R2_OOS_Roll20': m_eval['R2_OOS_Roll20'],
                    'Hist_Exp_Series': 'DL-base (MSE=2.689570e-06)',
                    'Pred_Std': m_eval['Pred_Std'],
                    'Pred_Corr': m_eval['Pred_Corr'],
                    'Bias_Level': bias_lvl,
                    'Pct_Le_0': pct_le_0
                })
                
    df_per_seed = pd.DataFrame(per_seed_metrics)
    per_seed_file = os.path.join(OUTPUT_DIR, "v6_metrics_per_seed.csv")
    df_per_seed.to_csv(per_seed_file, index=False)
    print(f"✅ Tabel metrik per-seed tersimpan: {per_seed_file}")
    
    summary_list = []
    num_cols = ['MSE', 'MAE', 'R2_OOS_HistExp', 'Bias_Level', 'Pct_Le_0']
    for (m_name, fset_name, h_step), grp in df_per_seed.groupby(['Model', 'FeatureSet', 'Horizon']):
        rec = {'Model': m_name, 'FeatureSet': fset_name, 'Horizon': h_step, 'N_Seeds': len(grp)}
        for c in num_cols:
            rec[f'{c}_mean'] = grp[c].mean()
            rec[f'{c}_std']  = grp[c].std()
            rec[f'{c}_min']  = grp[c].min()
            rec[f'{c}_max']  = grp[c].max()
        summary_list.append(rec)
        
    df_seed_summary = pd.DataFrame(summary_list)
    seed_summary_file = os.path.join(OUTPUT_DIR, "v6_metrics_seed_summary.csv")
    df_seed_summary.to_csv(seed_summary_file, index=False)
    print(f"✅ Tabel ringkasan statistik benih tersimpan: {seed_summary_file}")
    
    ensemble_dfs = {}
    for (m_name, fset_name, tgt) in CONFIGS:
        cfg_base = f"{m_name}_{fset_name}_{tgt}"
        template_df = raw_preds[f"{cfg_base}_s{SEEDS[0]}"][['unique_id', 'ds', 'cutoff', 'real_date', 'y']].copy()
        
        seed_preds_mat = []
        for s in SEEDS:
            seed_preds_mat.append(raw_preds[f"{cfg_base}_s{s}"][m_name].values)
            
        template_df[m_name] = np.mean(seed_preds_mat, axis=0)
        ensemble_dfs[cfg_base] = template_df
        
    ensemble_export_df = pd.DataFrame()
    for cfg_k, e_df in ensemble_dfs.items():
        sub_e = e_df[['cutoff', 'real_date', 'unique_id', 'ds', 'y', cfg_k.split('_')[0]]].copy()
        sub_e.columns = ['cutoff', 'real_date', 'unique_id', 'ds', 'y', 'prediction']
        sub_e['Config'] = cfg_k
        ensemble_export_df = pd.concat([ensemble_export_df, sub_e], ignore_index=True)
    ensemble_pred_file = os.path.join(OUTPUT_DIR, "v6_ensemble_predictions.csv")
    ensemble_export_df.to_csv(ensemble_pred_file, index=False)
    print(f"✅ Prediksi ensemble tersimpan: {ensemble_pred_file}")
    
    ensemble_metrics = []
    for (m_name, fset_name, tgt) in CONFIGS:
        cfg_k = f"{m_name}_{fset_name}_{tgt}"
        e_df = ensemble_dfs[cfg_k]
        c_origins = v5_helpers.get_cutoff_origins(e_df, hist_series)
        
        for h_val, h_step in [(1, 'h=1'), (10, 'h=10')]:
            grp = e_df.groupby('cutoff').head(1) if h_val == 1 else e_df.groupby('cutoff').tail(1)
            yt = grp['y'].values
            yp = grp[m_name].values
            hexp_arr, roll_arr, _ = v5_helpers.compute_baselines(grp, hist_series, c_origins)
            
            m_eval = v5_helpers.evaluate_metrics(yt, yp, hexp_arr, roll_arr)
            bias_lvl = float(np.mean(yp) / np.mean(yt)) if np.mean(yt) > 0 else np.nan
            pct_le_0 = float(np.mean(yp <= 0) * 100.0)
            
            ensemble_metrics.append({
                'Config': cfg_k,
                'Model': m_name,
                'FeatureSet': fset_name,
                'Horizon': h_step,
                'N': m_eval['N'],
                'MSE': m_eval['MSE'],
                'MAE': m_eval['MAE'],
                'RMSE': m_eval['RMSE'],
                'R2_OOS_HistExp': m_eval['R2_OOS_HistExp'],
                'R2_OOS_Roll20': m_eval['R2_OOS_Roll20'],
                'Hist_Exp_Series': 'DL-base (MSE=2.689570e-06)',
                'Pred_Std': m_eval['Pred_Std'],
                'Pred_Corr': m_eval['Pred_Corr'],
                'Bias_Level': bias_lvl,
                'Pct_Le_0': pct_le_0
            })
    df_ens_metrics = pd.DataFrame(ensemble_metrics)
    ens_metrics_file = os.path.join(OUTPUT_DIR, "v6_metrics_ensemble.csv")
    df_ens_metrics.to_csv(ens_metrics_file, index=False)
    print(f"✅ Metrik evaluasi ensemble tersimpan: {ens_metrics_file}")
    
    dm_ensemble_records = []
    
    for (m_name, fset_name, tgt) in CONFIGS:
        cfg_k = f"{m_name}_{fset_name}_{tgt}"
        e_df = ensemble_dfs[cfg_k]
        c_origins = v5_helpers.get_cutoff_origins(e_df, hist_series)
        
        for h_val, h_step in [(1, 'h=1'), (10, 'h=10')]:
            grp = e_df.groupby('cutoff').head(1) if h_val == 1 else e_df.groupby('cutoff').tail(1)
            yt = grp['y'].values
            yp = grp[m_name].values
            hexp_arr, roll_arr, _ = v5_helpers.compute_baselines(grp, hist_series, c_origins)
            
            baselines = {
                'Hist_Exp (DL-base, MSE=2.689570e-06)': hexp_arr,
                'Roll_20 (DL-base, MSE=2.407078e-06)': roll_arr,
                'HAR (v2, MSE=2.234681e-06)': df_har_preds[h_step]['HAR'].values
            }
            
            for b_name, b_preds in baselines.items():
                dm_res = v5_helpers.diebold_mariano_test(
                    y_true=yt,
                    y_pred_m=yp,
                    y_pred_b=b_preds,
                    h=h_val,
                    m_family=M_FAMILY_BASELINES,
                    use_hln=True
                )
                dm_ensemble_records.append({
                    'Comparison_Type': 'Ensemble_vs_Baseline',
                    'Config': cfg_k,
                    'Model': m_name,
                    'FeatureSet': fset_name,
                    'Horizon': h_step,
                    'Benchmark': b_name,
                    'stat': dm_res['stat'],
                    'p_raw': dm_res['p_raw'],
                    'p_bonf': dm_res['p_bonf'],
                    'm_family': M_FAMILY_BASELINES,
                    'md_MSE': dm_res['md_MSE'],
                    'Winner': cfg_k if dm_res['p_bonf'] < 0.05 and dm_res['md_MSE'] < 0 else (b_name if dm_res['p_bonf'] < 0.05 and dm_res['md_MSE'] > 0 else 'Tie')
                })
                
    for m_name in MODELS:
        spill_k = f"{m_name}_SPILL_T0"
        nospill_k = f"{m_name}_NOSPILL_T0"
        e_spill = ensemble_dfs[spill_k]
        e_nospill = ensemble_dfs[nospill_k]
        
        for h_val, h_step in [(1, 'h=1'), (10, 'h=10')]:
            grp_sp = e_spill.groupby('cutoff').head(1) if h_val == 1 else e_spill.groupby('cutoff').tail(1)
            grp_no = e_nospill.groupby('cutoff').head(1) if h_val == 1 else e_nospill.groupby('cutoff').tail(1)
            
            yt = grp_sp['y'].values
            yp_sp = grp_sp[m_name].values
            yp_no = grp_no[m_name].values
            
            dm_abl = v5_helpers.diebold_mariano_test(
                y_true=yt,
                y_pred_m=yp_sp,
                y_pred_b=yp_no,
                h=h_val,
                m_family=M_FAMILY_ABLATION,
                use_hln=True
            )
            dm_ensemble_records.append({
                'Comparison_Type': 'Ablation_SPILL_vs_NOSPILL',
                'Config': f"{m_name}_SPILL_vs_NOSPILL",
                'Model': m_name,
                'FeatureSet': 'SPILL_vs_NOSPILL',
                'Horizon': h_step,
                'Benchmark': f"{m_name}_NOSPILL",
                'stat': dm_abl['stat'],
                'p_raw': dm_abl['p_raw'],
                'p_bonf': dm_abl['p_bonf'],
                'm_family': M_FAMILY_ABLATION,
                'md_MSE': dm_abl['md_MSE'],
                'Winner': 'SPILL' if dm_abl['p_bonf'] < 0.05 and dm_abl['md_MSE'] < 0 else ('NOSPILL' if dm_abl['p_bonf'] < 0.05 and dm_abl['md_MSE'] > 0 else 'Tie')
            })
            
    df_dm_ensemble = pd.DataFrame(dm_ensemble_records)
    dm_ensemble_file = os.path.join(OUTPUT_DIR, "v6_dm_ensemble.csv")
    df_dm_ensemble.to_csv(dm_ensemble_file, index=False)
    print(f"✅ Tabel uji DM ensemble tersimpan: {dm_ensemble_file}")
    
    dm_per_seed_records = []
    for seed in SEEDS:
        for (m_name, fset_name, tgt) in CONFIGS:
            cfg_k = f"{m_name}_{fset_name}_{tgt}_s{seed}"
            df_p = raw_preds[cfg_k]
            c_origins = v5_helpers.get_cutoff_origins(df_p, hist_series)
            
            for h_val, h_step in [(1, 'h=1'), (10, 'h=10')]:
                grp = df_p.groupby('cutoff').head(1) if h_val == 1 else df_p.groupby('cutoff').tail(1)
                yt = grp['y'].values
                yp = grp[m_name].values
                hexp_arr, roll_arr, _ = v5_helpers.compute_baselines(grp, hist_series, c_origins)
                
                baselines = {
                    'Hist_Exp (DL-base, MSE=2.689570e-06)': hexp_arr,
                    'Roll_20 (DL-base, MSE=2.407078e-06)': roll_arr,
                    'HAR (v2, MSE=2.234681e-06)': df_har_preds[h_step]['HAR'].values
                }
                for b_name, b_preds in baselines.items():
                    res_raw = v5_helpers.diebold_mariano_test(yt, yp, b_preds, h=h_val, m_family=1, use_hln=True)
                    dm_per_seed_records.append({
                        'Model': m_name,
                        'FeatureSet': fset_name,
                        'Seed': seed,
                        'Horizon': h_step,
                        'Benchmark': b_name,
                        'stat': res_raw['stat'],
                        'p_raw': res_raw['p_raw'],
                        'md_MSE': res_raw['md_MSE'],
                        'is_p_raw_lt_005': (res_raw['p_raw'] < 0.05 and res_raw['md_MSE'] < 0)
                    })
                    
    df_dm_per_seed = pd.DataFrame(dm_per_seed_records)
    dm_per_seed_file = os.path.join(OUTPUT_DIR, "v6_dm_per_seed.csv")
    df_dm_per_seed.to_csv(dm_per_seed_file, index=False)
    
    dm_summary_seed = df_dm_per_seed.groupby(['Model', 'FeatureSet', 'Horizon', 'Benchmark'])['is_p_raw_lt_005'].agg(
        n_seeds_outperform_p_raw_005='sum',
        total_seeds='count'
    ).reset_index()
    dm_summary_seed['ratio_str'] = dm_summary_seed['n_seeds_outperform_p_raw_005'].astype(str) + " / " + dm_summary_seed['total_seeds'].astype(str)
    dm_seed_sum_file = os.path.join(OUTPUT_DIR, "v6_dm_per_seed_summary.csv")
    dm_summary_seed.to_csv(dm_seed_sum_file, index=False)
    print(f"✅ Uji DM per-seed deskriptif tersimpan: {dm_per_seed_file} & {dm_seed_sum_file}")


=== STAGE D: AGREGASI, ENSEMBLE, & UJI DIEBOLD-MARIANO (CPU) ===
Berhasil memuat 20 file partial (5 seeds x 4 configs).
✅ Tabel metrik per-seed tersimpan: ./v6_outputs/v6_metrics_per_seed.csv
✅ Tabel ringkasan statistik benih tersimpan: ./v6_outputs/v6_metrics_seed_summary.csv
✅ Prediksi ensemble tersimpan: ./v6_outputs/v6_ensemble_predictions.csv
✅ Metrik evaluasi ensemble tersimpan: ./v6_outputs/v6_metrics_ensemble.csv
✅ Tabel uji DM ensemble tersimpan: ./v6_outputs/v6_dm_ensemble.csv
✅ Uji DM per-seed deskriptif tersimpan: ./v6_outputs/v6_dm_per_seed.csv & ./v6_outputs/v6_dm_per_seed_summary.csv


# CELL 9: MANIFEST OUTPUT, CHECKSUM MD5, & PACKAGING ZIP


In [9]:
# ======================================================================================
# CELL 9: MANIFEST OUTPUT, CHECKSUM MD5, & PACKAGING ZIP
# ======================================================================================

print("=== MANIFEST KELUARAN V6 & PACKAGING ZIP ===")

manifest_records = []
all_csvs = sorted(glob.glob(os.path.join(OUTPUT_DIR, "*.csv")) + glob.glob(os.path.join(OUTPUT_DIR, "*.json")))

for fp in all_csvs:
    fname = os.path.basename(fp)
    if fname.endswith("v6_manifest.csv") or fname.endswith("v6_outputs.zip"):
        continue
    f_size = os.path.getsize(fp)
    f_md5 = get_file_md5(fp)
    manifest_records.append({
        'filename': fname,
        'size_bytes': f_size,
        'md5': f_md5
    })

df_manifest = pd.DataFrame(manifest_records)
manifest_file = os.path.join(OUTPUT_DIR, "v6_manifest.csv")
df_manifest.to_csv(manifest_file, index=False)

zip_path = os.path.join(OUTPUT_DIR, "v6_outputs.zip")
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for fp in all_csvs:
        zipf.write(fp, arcname=os.path.basename(fp))
    zipf.write(manifest_file, arcname=os.path.basename(manifest_file))

print(f"\n✅ Arsip zip luaran berhasil dibuat: {zip_path} ({os.path.getsize(zip_path)/1024:.2f} KB)")
print(f"✅ Tabel Manifest ({len(df_manifest)} files):")
print(df_manifest.to_string(index=False))


=== MANIFEST KELUARAN V6 & PACKAGING ZIP ===

✅ Arsip zip luaran berhasil dibuat: ./v6_outputs/v6_outputs.zip (806.33 KB)
✅ Tabel Manifest (35 files):
                             filename  size_bytes                              md5
         v6_best_hyperparameters.json         359 571547282c2d4d0a6bd442fd1059552e
                   v6_dm_ensemble.csv        4744 8046ddf376391810185f508ea81a2223
                   v6_dm_per_seed.csv       14868 022c499f5f706656b0105d486abda0be
           v6_dm_per_seed_summary.csv        1594 ad531f332ef03781099caa17450c5c56
          v6_ensemble_predictions.csv     1186169 9e1bdd958db2414afed2c045773d0d78
                        v6_es_log.csv        2495 8679758ff1445d1ba6516e2ac50e2266
              v6_metrics_ensemble.csv        2074 284425d5bc469a61834be9f8ff27e503
              v6_metrics_per_seed.csv        9574 1db213141e995c9d5490cea2bcd74930
          v6_metrics_seed_summary.csv        3591 aaa656524147f48cc04475f917c429d9
v6_partial_NBEATSx_